In [ ]:
import pandas as pd
import numpy as np

# 1. Load your processed cluster dataset
clusters = pd.read_csv("final_cluster_features.csv") # The one with normalized features

# 2. Define the scoring formula
def calculate_score(user, cluster):
    # Preference Score (Weights)
    pref_score = (
        (user['Likes_Beach'] * cluster['Beach'] * 30) +
        (user['Likes_Mountain'] * (cluster['Nature'] + cluster['Hiking']) * 30) +
        (user['Likes_Culture'] * (cluster['History'] + cluster['Religious'] + cluster['Culture']) * 30) +
        (user['Likes_Adventure'] * (cluster['Adventure'] + cluster['Safari']) * 20)
    )
    
    # Distance/Days Logic
    # Users on short trips (1-2 days) prefer low distance (close to 0)
    # Users on long trips (5-7 days) can tolerate high distance (close to 1)
    ideal_dist = user['Total_Days'] / 7.0
    dist_score = (1 - abs(cluster['Avg_Distance'] - ideal_dist)) * 20
    
    # Budget Penalty (If budget is low (1), penalize high distance)
    budget_penalty = (cluster['Avg_Distance'] * 20) if user['Budget'] == 1 else 0
    
    # Random Noise
    noise = np.random.uniform(-5, 5)
    
    total_score = pref_score + dist_score - budget_penalty + noise
    return max(0, min(100, total_score))

# 3. Generate 6,000 samples
data = []
for _ in range(6000):
    # Random User Profile
    user = {
        'Likes_Beach': np.random.randint(0, 2),
        'Likes_Mountain': np.random.randint(0, 2),
        'Likes_Culture': np.random.randint(0, 2),
        'Likes_Adventure': np.random.randint(0, 2),
        'Budget': np.random.randint(1, 4), # 1=Low, 2=Med, 3=High
        'Total_Days': np.random.randint(1, 8)
    }
    
    # Score for each cluster
    for _, cluster in clusters.iterrows():
        row = user.copy()
        # Add cluster features
        row.update(cluster.drop('Cluster_Name').to_dict())
        # Calculate Score
        row['Score'] = calculate_score(user, cluster)
        data.append(row)

# 4. Save to CSV
train_df = pd.DataFrame(data)
train_df.to_csv("ml_training_data.csv", index=False)

print(f"Generated {len(train_df)} rows of training data.")
train_df.head()

Generated 108000 rows of training data.


,Likes_Beach,Likes_Mountain,Likes_Culture,Likes_Adventure,Budget,Total_Days,Adventure,Architecture,Beach,Birding,...,Nature,Park,Relax,Religious,Safari,Shopping,Viewpoint,Num_Places,Avg_Distance,Score
0,1,1,1,1,1,1,0.000000,0.021739,0.000000,0.021739,...,0.152174,0.021739,0.043478,0.173913,0.000000,0.021739,0.043478,0.625000,0.437584,33.656721
1,1,1,1,1,1,1,0.148148,0.037037,0.222222,0.000000,...,0.185185,0.037037,0.000000,0.148148,0.037037,0.037037,0.037037,0.229167,0.721508,14.162178
2,1,1,1,1,1,1,0.015625,0.046875,0.171875,0.031250,...,0.125000,0.062500,0.078125,0.218750,0.000000,0.046875,0.000000,1.000000,0.000000,34.437140
3,1,1,1,1,1,1,0.000000,0.018519,0.000000,0.000000,...,0.166667,0.000000,0.055556,0.018519,0.111111,0.000000,0.055556,0.791667,0.427961,29.279392
4,1,1,1,1,1,1,0.058824,0.039216,0.000000,0.000000,...,0.176471,0.000000,0.058824,0.078431,0.000000,0.000000,0.176471,0.729167,0.545308,26.493725


In [5]:
#train_df['Score'].sort_values

train_df

,Likes_Beach,Likes_Mountain,Likes_Culture,Likes_Adventure,Budget,Total_Days,Adventure,Architecture,Beach,Birding,...,Nature,Park,Relax,Religious,Safari,Shopping,Viewpoint,Num_Places,Avg_Distance,Score
0,1,1,1,1,1,1,0.000000,0.021739,0.000000,0.021739,...,0.152174,0.021739,0.043478,0.173913,0.000000,0.021739,0.043478,0.625000,0.437584,33.656721
1,1,1,1,1,1,1,0.148148,0.037037,0.222222,0.000000,...,0.185185,0.037037,0.000000,0.148148,0.037037,0.037037,0.037037,0.229167,0.721508,14.162178
2,1,1,1,1,1,1,0.015625,0.046875,0.171875,0.031250,...,0.125000,0.062500,0.078125,0.218750,0.000000,0.046875,0.000000,1.000000,0.000000,34.437140
3,1,1,1,1,1,1,0.000000,0.018519,0.000000,0.000000,...,0.166667,0.000000,0.055556,0.018519,0.111111,0.000000,0.055556,0.791667,0.427961,29.279392
4,1,1,1,1,1,1,0.058824,0.039216,0.000000,0.000000,...,0.176471,0.000000,0.058824,0.078431,0.000000,0.000000,0.176471,0.729167,0.545308,26.493725
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107995,1,1,1,1,3,6,0.046512,0.000000,0.441860,0.023256,...,0.139535,0.000000,0.000000,0.069767,0.023256,0.000000,0.069767,0.562500,0.473756,42.252609
107996,1,1,1,1,3,6,0.018519,0.000000,0.000000,0.000000,...,0.314815,0.037037,0.074074,0.018519,0.018519,0.018519,0.129630,0.791667,0.419585,31.814201
107997,1,1,1,1,3,6,0.157895,0.000000,0.263158,0.000000,...,0.105263,0.000000,0.000000,0.157895,0.105263,0.000000,0.000000,0.062500,0.919824,48.050390
107998,1,1,1,1,3,6,0.000000,0.000000,0.095238,0.047619,...,0.142857,0.000000,0.047619,0.190476,0.095238,0.000000,0.095238,0.104167,0.614522,41.704746
